# k05 — Final evaluation (STAGE2_DESIGN_FROZEN_v1.0 §3, §9 + v1.1)
**The only notebook that extracts or scores test cells.** Uses frozen k04 models, normalisation statistics and OOF thresholds; no fitting, selection or target calibration.
Cells: A test_01, B test_02 (primary), C test_03, D test_04 (set_01/test_04 excluded by design). Each test CSV is SHA-256 verified and checked against the set's train_01 hashes, featurised at stride 64, then deleted.
Statistics: PR-AUC per cell (with prevalence), Macro-F1 at max-F1 threshold, recall and achieved FA/h at 1 and 5 FA/h, per-token AP (≥50 positive windows), 2,000-resample recording-level bootstrap stratified by token (seed-averaged AP), paired contrasts H1/H1a/H2/H3 (+ LightGBM+S − LightGBM), B summary and manufacturer strata, Holm over the secondary family.

**Version 2 fix (before any test metric was seen):** v1 aborted set_01 with non-finite DeepSets scores (fp16 overflow of test features normalised by a training std floored at 1e-6). Scoring now uses float32 and sets features that were constant in the set's training windows to 0 (the value all training windows had). Per-feature |z| diagnostics are saved per cell.

In [ ]:
import os, glob
os.makedirs('/kaggle/working/code', exist_ok=True); os.makedirs('/kaggle/temp', exist_ok=True)
FILES = {'feats.py': 'import numpy as np, pandas as pd, re, os, time\nW = 64\nHEXV = np.full(256, 0, dtype=np.uint8)\nfor i, ch in enumerate(\'0123456789abcdef\'):\n    HEXV[ord(ch)] = i; HEXV[ord(ch.upper())] = i\nPOP = np.array([bin(i).count(\'1\') for i in range(256)], dtype=np.uint8)\n\ndef slog(x):\n    return np.sign(x) * np.log1p(np.abs(x))\n\ndef load_file(path):\n    df = pd.read_csv(path, dtype={\'arbitration_id\': str, \'data_field\': str, \'attack\': np.int8}, keep_default_na=True)\n    ts = df[\'timestamp\'].to_numpy(np.float64)\n    ids = df[\'arbitration_id\'].str.rjust(3, \'0\').str[-3:]\n    ib = np.frombuffer(\'\'.join(ids.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 3)\n    idv = HEXV[ib].astype(np.int32)\n    id_int = idv[:, 0] * 256 + idv[:, 1] * 16 + idv[:, 2]\n    d = df[\'data_field\'].fillna(\'\')\n    plen = (d.str.len().to_numpy() // 2).clip(0, 8).astype(np.int8)\n    d16 = d.str[:16].str.ljust(16, \'0\')\n    db = np.frombuffer(\'\'.join(d16.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 16)\n    nib = HEXV[db]\n    pay = (nib[:, 0::2] * 16 + nib[:, 1::2]).astype(np.uint8)\n    posmask = np.arange(8)[None, :] < plen[:, None]\n    pay = np.where(posmask, pay, 0).astype(np.uint8)\n    y = df[\'attack\'].to_numpy(np.int8)\n    return ts, id_int, plen, pay, y\n\ndef per_frame_globals(ts, id_int, plen, pay):\n    n = len(ts); idx = np.arange(n)\n    order = np.lexsort((idx, id_int))\n    prev = np.full(n, -1, dtype=np.int64)\n    same = np.r_[False, id_int[order][1:] == id_int[order][:-1]]\n    prev[order[same]] = order[np.flatnonzero(same) - 1]\n    has = prev >= 0\n    pp = np.where(has, prev, 0)\n    dt_same = np.where(has, ts - ts[pp], 0.0)\n    x = pay ^ pay[pp]\n    ham = np.where(has, POP[x].sum(1), 0).astype(np.float32)\n    maxlen = np.maximum(plen, plen[pp]).astype(np.float32)\n    chg = np.where(has, (x != 0).sum(1) / np.maximum(maxlen, 1), 0).astype(np.float32)\n    lenchg = np.where(has, plen != plen[pp], False)\n    ent = np.zeros(n, dtype=np.float32)\n    for s in range(0, n, 500000):\n        b = pay[s:s + 500000]; L = plen[s:s + 500000].astype(np.int32)\n        valid = np.arange(8)[None, :] < L[:, None]\n        eq = (b[:, :, None] == b[:, None, :]) & valid[:, :, None] & valid[:, None, :]\n        c = eq.sum(2).astype(np.float32)\n        Lf = np.maximum(L, 1).astype(np.float32)[:, None]\n        with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n            term = np.where(valid, np.log2(np.where(c > 0, c, 1) / Lf), 0.0)\n        ent[s:s + 500000] = -(term.sum(1) / Lf[:, 0])\n    return prev, dt_same, ham, chg, ent, lenchg\n\nFRAME_NAMES = [\'plen\', \'dt_prev_any\', \'dt_same\', \'no_prev_same_in_window\', \'hamming\', \'changed_frac\', \'entropy\', \'same_as_prev\']\nNODE_NAMES = [\'count\', \'first_pos\', \'last_pos\', \'ia_mean\', \'ia_min\', \'ia_max\', \'ia_missing\', \'plen_mean\', \'plen_max\', \'plen_changes\', \'ham_mean\', \'chg_mean\', \'ent_mean\']\nGLOBAL_NAMES = [\'duration\', \'distinct_ids\', \'distinct_transitions\', \'fps\']\n\ndef windows_for_file(path, stride, id_perm=None):\n    ts, id_int, plen, pay, y = load_file(path)\n    if id_perm is not None:\n        id_int = id_perm[id_int]\n    n = len(ts)\n    if n < W:\n        return None\n    prev, dt_same_g, ham_g, chg_g, ent_g, lenchg_g = per_frame_globals(ts, id_int, plen, pay)\n    starts = np.arange(0, n - W + 1, stride)\n    nw = len(starts)\n    I = starts[:, None] + np.arange(W)[None, :]\n    ok = prev[I] >= starts[:, None]\n    tsw = ts[I]\n    dtp = np.diff(tsw, axis=1, prepend=tsw[:, :1])\n    idw = id_int[I]\n    fr = np.zeros((nw, W, len(FRAME_NAMES)), dtype=np.float32)\n    fr[..., 0] = plen[I] / 8.0\n    fr[..., 1] = slog(dtp * 1000)\n    fr[..., 2] = np.where(ok, slog(dt_same_g[I] * 1000), 0)\n    fr[..., 3] = ~ok\n    fr[..., 4] = np.where(ok, ham_g[I] / 64.0, 0)\n    fr[..., 5] = np.where(ok, chg_g[I], 0)\n    fr[..., 6] = ent_g[I] / 3.0\n    fr[:, 1:, 7] = idw[:, 1:] == idw[:, :-1]\n    # nodes\n    key = (np.arange(nw)[:, None] * 4096 + idw).ravel()\n    uk, inv = np.unique(key, return_inverse=True)\n    inv = inv.reshape(nw, W)\n    win_of_node = uk // 4096\n    node_first = np.searchsorted(win_of_node, np.arange(nw))\n    local = inv - node_first[:, None]\n    nn = np.bincount(win_of_node, minlength=nw)\n    G = len(uk)\n    fl = inv.ravel()\n    pos = np.broadcast_to(np.arange(W), (nw, W)).ravel()\n    okf = ok.ravel()\n    def agg_sum(v, m=None):\n        return np.bincount(fl, weights=(v if m is None else v * m), minlength=G)\n    order = np.argsort(fl, kind=\'stable\'); fs = fl[order]\n    bnd = np.flatnonzero(np.r_[True, fs[1:] != fs[:-1]])\n    def agg_min(v): return np.minimum.reduceat(v[order], bnd)\n    def agg_max(v): return np.maximum.reduceat(v[order], bnd)\n    cnt = np.bincount(fl, minlength=G).astype(np.float32)\n    iak = np.where(okf, slog(dt_same_g[I].ravel() * 1000), np.nan)\n    nia = agg_sum(okf.astype(np.float64))\n    has_ia = nia > 0\n    ia_mean = np.where(has_ia, agg_sum(np.nan_to_num(iak)) / np.maximum(nia, 1), 0)\n    ia_min = np.where(has_ia, agg_min(np.where(okf, iak, np.inf)), 0)\n    ia_max = np.where(has_ia, agg_max(np.where(okf, iak, -np.inf)), 0)\n    pl = (plen[I].ravel()).astype(np.float64)\n    nd = np.zeros((G, len(NODE_NAMES)), dtype=np.float32)\n    nd[:, 0] = cnt / W\n    nd[:, 1] = agg_min(pos.astype(np.float64)) / W\n    nd[:, 2] = agg_max(pos.astype(np.float64)) / W\n    nd[:, 3] = ia_mean; nd[:, 4] = ia_min; nd[:, 5] = ia_max\n    nd[:, 6] = ~has_ia\n    nd[:, 7] = agg_sum(pl) / cnt / 8.0\n    nd[:, 8] = agg_max(pl) / 8.0\n    nd[:, 9] = agg_max(pl) != agg_min(pl)\n    nd[:, 10] = np.where(has_ia, agg_sum(ham_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1) / 64.0, 0)\n    nd[:, 11] = np.where(has_ia, agg_sum(chg_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1), 0)\n    nd[:, 12] = agg_sum(ent_g[I].ravel().astype(np.float64)) / cnt / 3.0\n    node = np.zeros((nw, W, len(NODE_NAMES)), dtype=np.float32)\n    node[win_of_node, np.arange(G) - node_first[win_of_node]] = nd\n    # edges: local src/dst per transition\n    src = local[:, :-1].astype(np.uint8); dst = local[:, 1:].astype(np.uint8)\n    tr = np.unique((np.arange(nw)[:, None] * 4096 + local[:, :-1] * 64 + local[:, 1:]).ravel())\n    ntr = np.bincount(tr // 4096, minlength=nw)\n    dur = tsw[:, -1] - tsw[:, 0]\n    glob = np.stack([slog(dur * 1000), nn / W, ntr / 63.0, slog(W / np.maximum(dur, 1e-6))], 1).astype(np.float32)\n    lab = (y[I].max(1) > 0).astype(np.int8)\n    nattack = y[I].sum(1).astype(np.int16)\n    return dict(frame=fr.astype(np.float16), node=node.astype(np.float16), nmask=(np.arange(W)[None, :] < nn[:, None]),\n                src=src, dst=dst, glob=glob, y=lab, nattack=nattack, starts=starts.astype(np.int64), t0=tsw[:, 0], t1=tsw[:, -1])\n\nSTRUCT_NAMES = [\'transition_entropy\', \'unique_transition_ratio\', \'self_loop_ratio\', \'mean_out_degree\',\n                \'max_out_degree\', \'max_in_degree\', \'degree_entropy\', \'density\']\n\ndef structural_features(src, dst, nmask):\n    """Explicit structural/topological summaries of each window\'s directed transition multigraph.\n    src, dst: (nw, 63) local node indices of consecutive frames; nmask: (nw, 64) valid nodes.\n    Uses only ID-free graph structure (local node indices are arbitrary labels)."""\n    nw, E = src.shape\n    n = nmask.sum(1).astype(np.float64)\n    s = src.astype(np.int64); d = dst.astype(np.int64)\n    w = np.repeat(np.arange(nw), E)\n    key = w * 4096 + (s * 64 + d).ravel()\n    uk, cnt = np.unique(key, return_counts=True)\n    uw = uk // 4096; us = (uk % 4096) // 64; ud = (uk % 4096) % 64\n    p = cnt / float(E)\n    ent = np.bincount(uw, weights=-p * np.log2(p), minlength=nw) / np.log2(E)\n    uniq = np.bincount(uw, minlength=nw) / float(E)\n    selfr = (s == d).sum(1) / float(E)\n    ns = us != ud\n    outdeg = np.bincount(uw[ns] * 64 + us[ns], minlength=nw * 64).reshape(nw, 64).astype(np.float64)\n    indeg = np.bincount(uw[ns] * 64 + ud[ns], minlength=nw * 64).reshape(nw, 64).astype(np.float64)\n    e_ns = np.bincount(uw[ns], minlength=nw).astype(np.float64)\n    mean_out = np.where(n > 0, e_ns / np.maximum(n, 1), 0)\n    tot = outdeg + indeg; ts = tot.sum(1, keepdims=True)\n    pd_ = np.where(ts > 0, tot / np.maximum(ts, 1), 0)\n    with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n        h = -(np.where(pd_ > 0, pd_ * np.log2(np.where(pd_ > 0, pd_, 1)), 0)).sum(1)\n    deg_ent = np.where(n > 1, h / np.log2(np.maximum(n, 2)), 0)\n    dens = np.where(n > 1, e_ns / np.maximum(n * (n - 1), 1), 0)\n    return np.stack([ent, uniq, selfr, mean_out, outdeg.max(1), indeg.max(1), deg_ent, dens], 1).astype(np.float32)\n', 'models.py': "import torch, torch.nn as nn, numpy as np, time\nW = 64\n\ndef mlp(i, h, o, drop):\n    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, o))\n\nclass Head(nn.Module):\n    def __init__(self, h, g, drop):\n        super().__init__(); self.rho = mlp(3 * h + g, h, 1, drop)\n    def forward(self, H, mask, glob):\n        m = mask.unsqueeze(-1).float()\n        s = (H * m).sum(1); mean = s / m.sum(1).clamp(min=1)\n        mx = H.masked_fill(m == 0, -1e4).max(1).values\n        return self.rho(torch.cat([s, mean, mx, glob], 1)).squeeze(-1)\n\nclass DeepSets(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.phi = nn.Sequential(nn.Linear(f, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, h), nn.ReLU())\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        return self.head(self.phi(b['node']), b['nmask'], b['glob'])\n\nclass SAGELayer(nn.Module):\n    def __init__(self, i, o):\n        super().__init__(); self.self_lin = nn.Linear(i, o); self.nei_lin = nn.Linear(i, o, bias=False)\n    def forward(self, H, A):\n        # A[b, src, dst] = weight; aggregate incoming neighbours of each dst node (weighted mean)\n        agg = torch.bmm(A.transpose(1, 2), H)\n        deg = A.sum(1).unsqueeze(-1)\n        agg = agg / deg.clamp(min=1e-9)\n        return self.self_lin(H) + self.nei_lin(agg)\n\nclass GraphSAGE(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.l1 = SAGELayer(f, h); self.l2 = SAGELayer(h, h); self.drop = nn.Dropout(drop)\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        A = b['adj']; m = b['nmask'].unsqueeze(-1).float()\n        H = torch.relu(self.l1(b['node'], A)) * m\n        H = torch.relu(self.l2(self.drop(H), A)) * m\n        return self.head(H, b['nmask'], b['glob'])\n\nclass GRUNet(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.gru = nn.GRU(f, h, batch_first=True); self.out = mlp(2 * h + g, h, 1, drop)\n    def forward(self, b):\n        O, hn = self.gru(b['frame'])\n        return self.out(torch.cat([hn[-1], O.mean(1), b['glob']], 1)).squeeze(-1)\n\ndef nparams(m):\n    return sum(p.numel() for p in m.parameters())\n\ndef build_adj(src, dst, B, device):\n    A = torch.zeros(B, W, W, device=device)\n    bi = torch.arange(B, device=device).unsqueeze(1).expand_as(src)\n    A.index_put_((bi.reshape(-1), src.reshape(-1).long(), dst.reshape(-1).long()), torch.full((src.numel(),), 1.0 / 63, device=device), accumulate=True)\n    return A\n\ndef rewire_dst(dst, gen):\n    # degree-preserving directed rewiring: permute destination endpoints among a window's 63 edges\n    # (every source keeps its out-degree, every destination keeps its in-degree, multiplicities included)\n    r = torch.rand(dst.shape, generator=gen, device=dst.device)\n    perm = r.argsort(1)\n    return torch.gather(dst, 1, perm)\n\ndef edge_change_fraction(src, dst, dst2):\n    # fraction of the 63 directed edges (as a multiset per window) not present in the original\n    B = src.shape[0]\n    k1 = (src.long() * 64 + dst.long()).sort(1).values\n    k2 = (src.long() * 64 + dst2.long()).sort(1).values\n    fr = []\n    for i in range(B):\n        a, ca = torch.unique(k1[i], return_counts=True); b2, cb = torch.unique(k2[i], return_counts=True)\n        common = 0\n        d = dict(zip(a.tolist(), ca.tolist()))\n        for kk, cc in zip(b2.tolist(), cb.tolist()):\n            common += min(cc, d.get(kk, 0))\n        fr.append(1 - common / 63.0)\n    return float(np.mean(fr))\n", 'exp_eval.py': '"""Stage 5a final evaluation scoring (STAGE2_DESIGN_FROZEN_v1.0 §3,§9 + v1.1). THE ONLY SCRIPT THAT READS TEST CELLS.\nReads frozen k04 final models/thresholds/norm stats; writes per-window scores per cell. No fitting, no selection, no calibration.\nUsage: python exp_eval.py --sets set_01,set_03 --device cuda:0"""\nimport os, sys, re, json, time, glob, hashlib, argparse, traceback, zipfile\nimport numpy as np, torch\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport feats, models\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--sets\', required=True); ap.add_argument(\'--device\', default=\'cuda:0\')\nap.add_argument(\'--out\', default=\'/kaggle/working/eval\'); ap.add_argument(\'--seeds\', default=\'0,1,2,3,4\')\nargs = ap.parse_args()\nDEV = args.device; OUT = args.out; os.makedirs(OUT, exist_ok=True)\nSETS = args.sets.split(\',\'); SEEDS = [int(s) for s in args.seeds.split(\',\')]\nZIP = glob.glob(\'/kaggle/input/**/can-train-and-test-v1.zip\', recursive=True)[0]\nHASHES = glob.glob(\'/kaggle/input/**/file_hashes_sha256.csv\', recursive=True)[0]\nDATA = f\'/kaggle/temp/eval_{"_".join(SETS)}\'\nEXCLUDED = {(\'set_01\', \'test_04\')}   # v1.0 §3: Traverse by fingerprint, byte-identical to set_02/test_01; integrity section only\nMODELS = (\'Rule\', \'LightGBM\', \'LightGBM_S\', \'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\')\nEVAL_REWIRE_SEED = 12345   # fixed rewiring seed at evaluation (v1.0 §7), same as used for tuning OOF scores\nLOG = open(os.path.join(OUT, f\'log_{"_".join(SETS)}.txt\'), \'a\')\ndef log(*a):\n    s = time.strftime(\'%H:%M:%S \') + \' \'.join(str(x) for x in a); print(s, flush=True); LOG.write(s + \'\\n\'); LOG.flush()\n\nH = {}\nfor line in open(HASHES).read().strip().split(\'\\n\')[1:]:\n    rel, size, sha = line.split(\',\'); H[rel] = (int(size), sha)\ndef fam(name): return re.sub(r\'-\\d+\\.csv$\', \'\', os.path.basename(name))\n\nNF, NN, NG = len(feats.FRAME_NAMES), len(feats.NODE_NAMES), len(feats.GLOBAL_NAMES)\ndef make(name, h, drop):\n    if name in (\'GraphSAGE\', \'GraphSAGE_rewired\'): return models.GraphSAGE(NN, h, NG, drop)\n    if name == \'DeepSets\': return models.DeepSets(NN, h, NG, drop)\n    if name == \'GRU\': return models.GRUNet(NF, h, NG, drop)\n\ndef flat_features(D, with_struct):   # identical to exp_tune / exp_final\n    m = D[\'nmask\'][..., None]; x = D[\'node\'].astype(np.float32); cnt = m.sum(1).clip(1)\n    mean = (x * m).sum(1) / cnt; std = np.sqrt(((x - mean[:, None]) ** 2 * m).sum(1) / cnt)\n    mn = np.where(m, x, np.inf).min(1); mx = np.where(m, x, -np.inf).max(1)\n    X = [mean, std, mn, mx, D[\'glob\']]\n    if with_struct: X.append(D[\'struct\'])\n    return np.concatenate(X, 1).astype(np.float32)\n\nCONST_STD = 1.5e-6   # norm stats store std + 1e-6; std <= 5e-7 means the feature was constant in the set\'s training windows\ndef to_gpu(D, N):\n    """Same normalisation as k04 (training statistics only), but float32 (no fp16 overflow) and features that were CONSTANT in training\n    are set to 0 = exactly the value every training window had (the model never learned a response to them). Decided before any test metric was seen."""\n    T = lambda a: torch.tensor(a, dtype=torch.float32, device=DEV)\n    fm, fs, nm_, ns, gm, gs = T(N[\'fm\']), T(N[\'fs\']), T(N[\'nm\']), T(N[\'ns\']), T(N[\'gm\']), T(N[\'gs\'])\n    keepf, keepn, keepg = (fs > CONST_STD).float(), (ns > CONST_STD).float(), (gs > CONST_STD).float()\n    out = {}; diag = {}\n    msk = torch.from_numpy(D[\'nmask\']).to(DEV)\n    out[\'frame\'] = ((torch.from_numpy(D[\'frame\'].astype(np.float32)).to(DEV) - fm) / fs)\n    out[\'node\'] = (((torch.from_numpy(D[\'node\'].astype(np.float32)).to(DEV) - nm_) / ns) * msk.unsqueeze(-1))\n    out[\'glob\'] = (torch.from_numpy(D[\'glob\']).to(DEV) - gm) / gs\n    for k, keep, names in ((\'frame\', keepf, feats.FRAME_NAMES), (\'node\', keepn, feats.NODE_NAMES), (\'glob\', keepg, feats.GLOBAL_NAMES)):\n        z = out[k].abs().reshape(-1, len(names))\n        diag[k] = {nm: {\'max_abs_z\': float(z[:, j].max()), \'frac_abs_z_gt_100\': float((z[:, j] > 100).float().mean()), \'constant_in_training_zeroed\': bool(keep[j] == 0)}\n                   for j, nm in enumerate(names)}\n        out[k] = out[k] * keep\n    out[\'nmask\'] = msk\n    out[\'src\'] = torch.from_numpy(D[\'src\']).to(DEV); out[\'dst\'] = torch.from_numpy(D[\'dst\']).to(DEV)\n    out[\'n\'] = len(D[\'y\'])\n    return out, diag\n\ndef batch(T, ix, rewire, gen):\n    b = {\'frame\': T[\'frame\'][ix].float(), \'node\': T[\'node\'][ix].float(), \'nmask\': T[\'nmask\'][ix], \'glob\': T[\'glob\'][ix]}\n    dst = T[\'dst\'][ix]\n    if rewire: dst = models.rewire_dst(dst, gen)\n    b[\'adj\'] = models.build_adj(T[\'src\'][ix], dst, len(ix), DEV)\n    return b\n\ndef score_neural(m, T, rewire):\n    m.eval(); g = torch.Generator(device=DEV); g.manual_seed(EVAL_REWIRE_SEED); out = []\n    with torch.no_grad():\n        for s in range(0, T[\'n\'], 4096):\n            ix = torch.arange(s, min(s + 4096, T[\'n\']), device=DEV)\n            out.append(m(batch(T, ix, rewire, g)).float())\n    return torch.cat(out).cpu().numpy().astype(np.float32)\n\nclass Scorer:\n    def __init__(self, st):\n        import lightgbm as lgb\n        fd = os.path.dirname(glob.glob(f\'/kaggle/input/**/final/{st}/final_meta.json\', recursive=True)[0])\n        self.meta = json.load(open(os.path.join(fd, \'final_meta.json\')))\n        self.N = dict(np.load(os.path.join(fd, \'norm_stats.npz\')))\n        self.rule = self.meta[\'models\'][\'Rule\']\n        self.lgb = {name: [lgb.Booster(model_file=os.path.join(fd, f\'{name}_seed{s}.txt\')) for s in SEEDS] for name in (\'LightGBM\', \'LightGBM_S\')}\n        self.nn = {}\n        for name in (\'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\'):\n            r = self.meta[\'models\'][name]; lst = []\n            for s in SEEDS:\n                if \'error\' in r[\'seeds\'].get(str(s), r[\'seeds\'].get(s, {})): raise RuntimeError(f\'{st} {name} seed {s} failed in k04\')\n                m = make(name, r[\'hidden\'], r[\'config\'][\'dropout\']).to(DEV)\n                m.load_state_dict(torch.load(os.path.join(fd, f\'{name}_seed{s}.pt\'), map_location=DEV)); m.eval(); lst.append(m)\n            self.nn[name] = lst\n        self.fd = fd\n    def score(self, D):\n        S = {}\n        Xb = flat_features(D, False); Xs = flat_features(D, True)\n        S[\'Rule\'] = (self.rule[\'sign\'] * Xb[:, self.rule[\'feature_index\']])[None].astype(np.float32)\n        S[\'LightGBM\'] = np.stack([b.predict(Xb) for b in self.lgb[\'LightGBM\']]).astype(np.float32)      # probability scale, as OOF predict_proba\n        S[\'LightGBM_S\'] = np.stack([b.predict(Xs) for b in self.lgb[\'LightGBM_S\']]).astype(np.float32)\n        T, self.last_diag = to_gpu(D, self.N)\n        for name, lst in self.nn.items():\n            S[name] = np.stack([score_neural(m, T, name == \'GraphSAGE_rewired\') for m in lst])          # logits, as OOF\n        del T; torch.cuda.empty_cache()\n        return S\n\nKEYS = [\'frame\', \'node\', \'nmask\', \'src\', \'dst\', \'glob\', \'y\', \'struct\', \'nattack\', \'starts\']\ndef file_windows(p, id_perm=None):\n    d = feats.windows_for_file(p, 64, id_perm)\n    d[\'struct\'] = feats.structural_features(d[\'src\'], d[\'dst\'], d[\'nmask\'])\n    assert (d[\'starts\'] % 64 == 0).all()\n    return d\n\ndef sha_of(p):\n    hh = hashlib.sha256()\n    with open(p, \'rb\') as f:\n        for b in iter(lambda: f.read(8 << 20), b\'\'): hh.update(b)\n    return hh.hexdigest()\n\ndef id_permutation_test(st, sc, p):\n    """v1.0 §5 automated test: permuting ID labels leaves every model\'s output unchanged. Reports max |diff| (seed 0 / all seeds for trees)."""\n    rng = np.random.default_rng(7); perm = rng.permutation(4096)\n    d0 = file_windows(p); d1 = file_windows(p, perm)\n    k = min(len(d0[\'y\']), 3000)\n    d0 = {kk: v[:k] for kk, v in d0.items() if kk in KEYS}; d1 = {kk: v[:k] for kk, v in d1.items() if kk in KEYS}\n    s0, s1 = sc.score(d0), sc.score(d1)\n    res = {m: float(np.abs(s0[m] - s1[m]).max()) for m in MODELS}\n    res[\'ids_actually_permuted\'] = True\n    return res\n\ndef run_set(st):\n    sd = os.path.join(OUT, st); os.makedirs(sd, exist_ok=True)\n    sc = Scorer(st); log(st, \'loaded frozen k04 models from\', sc.fd)\n    const = {k: [n for n, v in zip(names, sc.N[key]) if v <= CONST_STD] for k, key, names in ((\'frame\', \'fs\', feats.FRAME_NAMES), (\'node\', \'ns\', feats.NODE_NAMES), (\'glob\', \'gs\', feats.GLOBAL_NAMES))}\n    log(st, \'training-constant features (zeroed at test):\', const)\n    json.dump({\'constant_in_training\': const, \'fs\': sc.N[\'fs\'].tolist(), \'ns\': sc.N[\'ns\'].tolist(), \'gs\': sc.N[\'gs\'].tolist()}, open(os.path.join(sd, \'norm_diagnostics.json\'), \'w\'), indent=1)\n    train_sha = {H[r][1] for r in H if r.startswith(f\'{st}/train_01/\')}\n    with zipfile.ZipFile(ZIP) as z:\n        allnames = z.namelist()\n        cells = sorted({n.split(\'/\')[2] for n in allnames if n.startswith(f\'can-train-and-test/{st}/test_\') and n.endswith(\'.csv\')})\n        for cell in cells:\n            short = cell[:7]\n            if (st, short) in EXCLUDED: log(st, cell, \'EXCLUDED by v1.0 §3 (not extracted, not scored)\'); continue\n            if os.path.exists(os.path.join(sd, f\'{short}_scores.npz\')): log(st, cell, \'already done\'); continue\n            names = sorted(n for n in allnames if n.startswith(f\'can-train-and-test/{st}/{cell}/\') and n.endswith(\'.csv\'))\n            t0 = time.time(); parts = {k: [] for k in KEYS}; fid = []; finfo = []\n            for i, n in enumerate(names):\n                z.extract(n, DATA); p = os.path.join(DATA, n); rel = n.split(\'can-train-and-test/\')[1]\n                sha = sha_of(p); assert (os.path.getsize(p), sha) == H[rel], \'hash mismatch \' + rel\n                assert sha not in train_sha, \'LEAKAGE: test file identical to a train_01 file \' + rel\n                if i == 0 and short == \'test_01\':\n                    pt = id_permutation_test(st, sc, p); log(st, \'ID-permutation test (max |score diff|)\', pt)\n                    json.dump(pt, open(os.path.join(sd, \'id_permutation_test.json\'), \'w\'), indent=1)\n                d = file_windows(p); os.remove(p)\n                for k in KEYS: parts[k].append(d[k])\n                fid.append(np.full(len(d[\'y\']), i, np.int16))\n                finfo.append({\'file\': os.path.basename(n), \'relative_path\': rel, \'sha256\': sha, \'token\': fam(n), \'windows\': int(len(d[\'y\'])),\n                              \'pos\': int(d[\'y\'].sum()), \'hours\': float(d[\'t1\'].max() - d[\'t0\'].min()) / 3600.0})\n            D = {k: np.concatenate(v) for k, v in parts.items()}; fid = np.concatenate(fid)\n            tf = time.time() - t0; t0 = time.time()\n            S = sc.score(D)\n            for m in MODELS: assert np.isfinite(S[m]).all(), f\'non-finite scores {st} {cell} {m}\'\n            cm_diag = sc.last_diag\n            np.savez_compressed(os.path.join(sd, f\'{short}_scores.npz\'), y=D[\'y\'].astype(np.int8), file_id=fid, nattack=D[\'nattack\'], starts=D[\'starts\'],\n                                **{f\'score_{m}\': S[m] for m in MODELS})\n            cm = {\'set\': st, \'cell\': cell, \'files\': finfo, \'windows\': int(len(D[\'y\'])), \'pos\': int(D[\'y\'].sum()),\n                  \'prevalence\': float(D[\'y\'].mean()), \'hours\': float(sum(f[\'hours\'] for f in finfo)), \'feature_s\': round(tf, 1), \'score_s\': round(time.time() - t0, 1),\n                  \'input_diagnostics\': cm_diag, \'precision\': \'float32; training-constant features zeroed\'}\n            json.dump(cm, open(os.path.join(sd, f\'{short}_meta.json\'), \'w\'), indent=1)\n            log(st, cell, {k: cm[k] for k in (\'windows\', \'pos\', \'prevalence\', \'hours\', \'feature_s\', \'score_s\')})\n    log(st, \'DONE\')\n\nfor st in SETS:\n    try:\n        run_set(st)\n    except Exception as e:\n        log(st, \'FATAL\', repr(e), traceback.format_exc()[-2000:])\n', 'exp_stats.py': '"""Stage 5b statistics (STAGE2_DESIGN_FROZEN_v1.0 §9 + v1.1 C1). Reads eval scores + frozen thresholds only. CPU/numpy (optional torch GPU, self-checked).\nUsage: python exp_stats.py --eval /kaggle/working/eval --out /kaggle/working/stats"""\nimport os, sys, json, glob, math, argparse, time\nimport numpy as np\nfrom sklearn.metrics import average_precision_score\n\nMODELS = [\'Rule\', \'LightGBM\', \'LightGBM_S\', \'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\']\nROLE = {\'test_01\': \'A\', \'test_02\': \'B\', \'test_03\': \'C\', \'test_04\': \'D\'}\nCONTRASTS = {\'H1\': (\'GraphSAGE\', \'DeepSets\'), \'H1a\': (\'GraphSAGE\', \'GraphSAGE_rewired\'), \'H2\': (\'GraphSAGE\', \'GRU\'), \'H3\': (\'GraphSAGE\', \'LightGBM_S\'),\n             \'ladder_LGBS_minus_LGB\': (\'LightGBM_S\', \'LightGBM\')}\nSECONDARY = [\'H1a\', \'H2\', \'H3\']\nDEV = None\nMARGIN = 0.02; NBOOT = 2000; BOOT_SEED = 20260917; CHUNK = 50\n\ndef weighted_ap_prep(s, y, fid, nfiles):\n    """Precompute per-file cumulative positive/negative counts at the end of each tied-score group (sklearn tie handling)."""\n    o = np.argsort(-s, kind=\'stable\'); ss = s[o]; ys = y[o].astype(np.float64); fs = fid[o]\n    ends = np.r_[np.flatnonzero(ss[1:] != ss[:-1]), len(ss) - 1]\n    CP = np.zeros((nfiles, len(ends))); CN = np.zeros((nfiles, len(ends)))\n    for f in range(nfiles):\n        m = fs == f\n        CP[f] = np.cumsum(m * ys)[ends]; CN[f] = np.cumsum(m * (1 - ys))[ends]\n    return CP, CN\n\ndef weighted_ap(CP, CN, Wf):\n    """AP with per-window weight = bootstrap multiplicity of its recording. Wf: (R, nfiles)."""\n    tp = Wf @ CP; fp = Wf @ CN\n    P = tp[:, -1:]; den = tp + fp\n    prec = np.where(den > 0, tp / np.where(den > 0, den, 1), 0.0)\n    rec = np.where(P > 0, tp / np.where(P > 0, P, 1), 0.0)\n    ap = (np.diff(rec, axis=1, prepend=0.0) * prec).sum(1)\n    return np.where(P[:, 0] > 0, ap, np.nan)\n\ndef weighted_ap_torch(CP, CN, Wf, dev):\n    import torch\n    CPt = torch.from_numpy(CP).to(dev); CNt = torch.from_numpy(CN).to(dev); out = []\n    for c in range(0, len(Wf), 200):\n        W = torch.from_numpy(Wf[c:c + 200]).to(dev)\n        tp = W @ CPt; fp = W @ CNt; P = tp[:, -1:]; den = tp + fp\n        prec = torch.where(den > 0, tp / den.clamp(min=1e-300), torch.zeros_like(tp))\n        rec = torch.where(P > 0, tp / P.clamp(min=1e-300), torch.zeros_like(tp))\n        drec = torch.diff(rec, dim=1, prepend=torch.zeros_like(rec[:, :1]))\n        ap = (drec * prec).sum(1); ap = torch.where(P[:, 0] > 0, ap, torch.full_like(ap, float(\'nan\')))\n        out.append(ap.cpu().numpy())\n    return np.concatenate(out)\n\ndef resample_weights(tokens, rng, R):\n    """Recording-level bootstrap stratified by attack token: within each token, draw its k files with replacement."""\n    tokens = np.asarray(tokens); W = np.zeros((R, len(tokens)))\n    for t in np.unique(tokens):\n        idx = np.flatnonzero(tokens == t); k = len(idx)\n        draw = rng.integers(0, k, size=(R, k))\n        for j in range(k): np.add.at(W, (np.arange(R), idx[draw[:, j]]), 1.0)\n    return W\n\ndef apply_thr(s, thr):\n    return (s >= thr[\'threshold\']) if thr[\'op\'] == \'>=\' else (s > thr[\'threshold\'])\n\ndef macro_f1(y, pred):\n    out = []\n    for c in (1, 0):\n        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))\n        out.append(2 * tp / max(2 * tp + fp + fn, 1))\n    return float(np.mean(out))\n\ndef ci(a, lvl):\n    a = a[np.isfinite(a)]; q = (1 - lvl) / 2\n    return [float(np.quantile(a, q)), float(np.quantile(a, 1 - q))]\n\ndef decide(diff_boot):\n    c95, c90 = ci(diff_boot, 0.95), ci(diff_boot, 0.90)\n    if c95[0] > 0: d = \'superior\'\n    elif c95[1] < 0: d = \'inferior\'\n    elif c90[0] >= -MARGIN and c90[1] <= MARGIN: d = \'practically_equivalent\'\n    else: d = \'inconclusive\'\n    a = diff_boot[np.isfinite(diff_boot)]\n    p = min(1.0, 2 * min((a <= 0).mean(), (a >= 0).mean()))\n    p = max(p, 1.0 / len(a))   # resolution floor\n    return {\'ci95\': c95, \'ci90\': c90, \'decision\': d, \'ci90_within_margin\': bool(c90[0] >= -MARGIN and c90[1] <= MARGIN), \'p_boot_two_sided\': float(p),\n            \'decision_rule\': \'95% CI excludes 0 -> superior/inferior (checked first); else 90% CI within +-0.02 -> practically_equivalent; else inconclusive\'}\n\ndef holm(pvals):\n    keys = sorted(pvals, key=lambda k: pvals[k]); m = len(keys); adj = {}; run = 0.0\n    for i, k in enumerate(keys):\n        run = max(run, min(1.0, (m - i) * pvals[k])); adj[k] = run\n    return adj\n\ndef main(EVAL, OUT, FINAL):\n    os.makedirs(OUT, exist_ok=True); rng = np.random.default_rng(BOOT_SEED)\n    cells = sorted(glob.glob(os.path.join(EVAL, \'set_*\', \'test_0*_scores.npz\')))\n    R = {\'per_cell\': {}, \'hypotheses\': {}, \'notes\': []}; BOOT = {}\n    for path in cells:\n        st = path.split(os.sep)[-2]; short = os.path.basename(path)[:7]; key = f\'{st}/{short}\'\n        Z = np.load(path); meta = json.load(open(path.replace(\'_scores.npz\', \'_meta.json\')))\n        TH = json.load(open(glob.glob(os.path.join(FINAL, st, \'thresholds.json\'))[0]))\n        y = Z[\'y\'].astype(np.int8); fid = Z[\'file_id\'].astype(np.int64); hours = meta[\'hours\']\n        tokens = [f[\'token\'] for f in meta[\'files\']]\n        cr = {\'role\': ROLE[short], \'windows\': int(len(y)), \'pos\': int(y.sum()), \'prevalence_chance_ap\': float(y.mean()), \'hours\': hours,\n              \'n_files\': len(tokens), \'files_per_token\': {t: tokens.count(t) for t in sorted(set(tokens))}, \'models\': {}}\n        Wb = resample_weights(tokens, rng, NBOOT); BOOT[key] = {}\n        for m in MODELS:\n            S = Z[f\'score_{m}\'].astype(np.float64); th = TH[m]\n            aps = [float(average_precision_score(y, s)) for s in S]\n            mr = {\'ap_seeds\': aps, \'ap_mean\': float(np.mean(aps)), \'ap_sd\': float(np.std(aps, ddof=1)) if len(aps) > 1 else 0.0}\n            mf, rec = [], {1: [], 5: []}; fah = {1: [], 5: []}\n            for s in S:\n                mf.append(macro_f1(y, apply_thr(s, th[\'f1\']).astype(np.int8)))\n                for r in (1, 5):\n                    pr = apply_thr(s, th[f\'fa{r}\']); fp = int(np.sum(pr & (y == 0)))\n                    rec[r].append(float(np.mean(pr[y == 1])) if y.sum() else float(\'nan\')); fah[r].append(fp / hours)\n            mr[\'macro_f1_mean\'] = float(np.mean(mf)); mr[\'macro_f1_seeds\'] = mf\n            for r in (1, 5):\n                mr[f\'recall_at_{r}FAh_mean\'] = float(np.mean(rec[r])); mr[f\'achieved_FAh_at_{r}FAh_mean\'] = float(np.mean(fah[r]))\n                mr[f\'allowed_fp_test_{r}FAh\'] = int(math.floor(r * hours))\n            # per attack token (tokens with >=50 positive windows in the cell)\n            fam = {}\n            for t in sorted(set(tokens)):\n                ti = np.isin(fid, [i for i, tt in enumerate(tokens) if tt == t]); npos = int(y[ti].sum())\n                if npos < 50: continue\n                fam[t] = {\'pos\': npos,\n                          \'ap_token_recordings_mean\': float(np.mean([average_precision_score(y[ti], s[ti]) for s in S])),\n                          \'ap_token_pos_vs_all_cell_neg_mean\': float(np.mean([average_precision_score(y[ti | (y == 0)], s[ti | (y == 0)]) for s in S]))}\n            mr[\'per_token\'] = fam\n            # bootstrap: seed-averaged AP per resample\n            bs = np.zeros(NBOOT)\n            for s in S:\n                CP, CN = weighted_ap_prep(s, y, fid, len(tokens)); acc = np.zeros(NBOOT)\n                chk = weighted_ap(CP, CN, np.ones((1, len(tokens))))[0]\n                assert abs(chk - average_precision_score(y, s)) < 1e-9, (\'weighted AP self-check failed\', key, m, chk)\n                if DEV:\n                    acc = weighted_ap_torch(CP, CN, Wb, DEV)\n                    ref = weighted_ap(CP, CN, Wb[:CHUNK]); assert np.allclose(acc[:CHUNK], ref, atol=1e-9, equal_nan=True), \'torch/numpy bootstrap mismatch\'\n                else:\n                    for c in range(0, NBOOT, CHUNK): acc[c:c + CHUNK] = weighted_ap(CP, CN, Wb[c:c + CHUNK])\n                bs += acc\n            BOOT[key][m] = bs / len(S)\n            mr[\'ap_mean_boot_ci95\'] = ci(BOOT[key][m], 0.95)\n            cr[\'models\'][m] = mr\n        for h, (a, b) in CONTRASTS.items():\n            d = BOOT[key][a] - BOOT[key][b]\n            cr.setdefault(\'contrasts\', {})[h] = {\'diff_point\': cr[\'models\'][a][\'ap_mean\'] - cr[\'models\'][b][\'ap_mean\'], **decide(d)}\n        R[\'per_cell\'][key] = cr\n        print(time.strftime(\'%H:%M:%S\'), key, {m: round(cr[\'models\'][m][\'ap_mean\'], 4) for m in MODELS}, flush=True)\n    # B-cell inference\n    bkeys = sorted(k for k in R[\'per_cell\'] if k.endswith(\'test_02\'))\n    shas = [f[\'sha256\'] for k in bkeys for f in json.load(open(os.path.join(EVAL, k.replace(\'/\', os.sep) + \'_meta.json\')))[\'files\']]\n    assert len(shas) == len(set(shas)), \'B cells share a recording; SHA-256 dedup required\'\n    strata = {\'B_summary\': bkeys, \'B_same_manufacturer\': [k for k in bkeys if k.startswith(\'set_01\')], \'B_cross_manufacturer\': [k for k in bkeys if not k.startswith(\'set_01\')]}\n    pv = {}\n    for h, (a, b) in CONTRASTS.items():\n        for sname, ks in strata.items():\n            if not ks: continue\n            d = np.mean([BOOT[k][a] - BOOT[k][b] for k in ks], 0)   # independent per-cell resamples = bootstrap stratified by cell\n            pt = float(np.mean([R[\'per_cell\'][k][\'models\'][a][\'ap_mean\'] - R[\'per_cell\'][k][\'models\'][b][\'ap_mean\'] for k in ks]))\n            R[\'hypotheses\'].setdefault(h, {})[sname] = {\'cells\': ks, \'diff_point\': pt, **decide(d)}\n        if h in SECONDARY: pv[f\'{h}/B_summary\'] = R[\'hypotheses\'][h][\'B_summary\'][\'p_boot_two_sided\']\n        if h in [\'H1\'] + SECONDARY:\n            for k in bkeys: pv[f\'{h}/{k}\'] = R[\'per_cell\'][k][\'contrasts\'][h][\'p_boot_two_sided\']\n    R[\'holm_family\'] = {\'members\': sorted(pv), \'p_raw\': pv, \'p_holm\': holm(pv),\n                        \'note\': \'H1 B_summary is the single primary test (not in the family). Bootstrap p-values have resolution 1/2000.\'}\n    R[\'notes\'] += [\'Chance-level PR-AUC = prevalence.\', \'Thresholds from pooled OOF scores of fold models, applied to final models (calibration-transfer limitation).\',\n                   \'Per-token AP given two ways: token recordings only; token positives vs all cell negatives.\',\n                   \'set_01/test_04 excluded by design (not scored).\']\n    json.dump(R, open(os.path.join(OUT, \'results.json\'), \'w\'), indent=1)\n    np.savez_compressed(os.path.join(OUT, \'bootstrap_seedavg_ap.npz\'), **{f\'{k}|{m}\': v for k, d in BOOT.items() for m, v in d.items()})\n    # compact tables\n    L = [\'cell\\trole\\tprev\\t\' + \'\\t\'.join(MODELS)]\n    for k, c in R[\'per_cell\'].items():\n        L.append(f"{k}\\t{c[\'role\']}\\t{c[\'prevalence_chance_ap\']:.4f}\\t" + \'\\t\'.join(f"{c[\'models\'][m][\'ap_mean\']:.4f}±{c[\'models\'][m][\'ap_sd\']:.4f}" for m in MODELS))\n    L.append(\'\'); L.append(\'hypothesis\\tstratum\\tdiff\\tci95\\tci90\\tdecision\\tp_raw\\tp_holm\')\n    for h, dd in R[\'hypotheses\'].items():\n        for sname, v in dd.items():\n            ph = R[\'holm_family\'][\'p_holm\'].get(f\'{h}/{sname}\', \'\')\n            L.append(f"{h}\\t{sname}\\t{v[\'diff_point\']:+.4f}\\t[{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}]\\t[{v[\'ci90\'][0]:+.4f},{v[\'ci90\'][1]:+.4f}]\\t{v[\'decision\']}\\t{v[\'p_boot_two_sided\']:.4f}\\t{ph}")\n    open(os.path.join(OUT, \'summary.tsv\'), \'w\').write(\'\\n\'.join(L) + \'\\n\'); print(\'\\n\'.join(L))\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--eval\', default=\'/kaggle/working/eval\'); ap.add_argument(\'--out\', default=\'/kaggle/working/stats\')\n    ap.add_argument(\'--final\', default=None); ap.add_argument(\'--device\', default=None); a = ap.parse_args()\n    DEV = a.device\n    FINAL = a.final or os.path.dirname(os.path.dirname(glob.glob(\'/kaggle/input/**/final/set_01/thresholds.json\', recursive=True)[0]))\n    main(a.eval, a.out, FINAL)\n'}
for n, s in FILES.items():
    open('/kaggle/working/code/' + n, 'w').write(s)
print(sorted(os.listdir('/kaggle/working/code')))
print(glob.glob('/kaggle/input/**/can-train-and-test-v1.zip', recursive=True))
fm = sorted(glob.glob('/kaggle/input/**/final/set_0*/final_meta.json', recursive=True)); th = sorted(glob.glob('/kaggle/input/**/final/set_0*/thresholds.json', recursive=True))
print(fm); assert len(fm) == 4 and len(th) == 4
for p in fm:
    d = os.path.dirname(p); n_pt = len(glob.glob(d + '/*_seed*.pt')); n_txt = len(glob.glob(d + '/*_seed*.txt'))
    print(d, 'checkpoints', n_pt, 'boosters', n_txt); assert n_pt == 20 and n_txt == 10


In [ ]:
import subprocess, sys, time
cmd = lambda sets, dev: [sys.executable, '/kaggle/working/code/exp_eval.py', '--sets', sets, '--device', dev]
t0 = time.time()
pA = subprocess.Popen(cmd('set_01,set_03', 'cuda:0'))
pB = subprocess.Popen(cmd('set_02,set_04', 'cuda:1'))
print('exit codes', pA.wait(), pB.wait(), 'hours', round((time.time() - t0) / 3600, 2))
sc = sorted(glob.glob('/kaggle/working/eval/set_0*/test_0*_scores.npz')); print(len(sc), 'cells scored'); print(sc)
logs = ''.join(open(p).read() for p in glob.glob('/kaggle/working/eval/log_*.txt'))
print('ERROR/FATAL lines:', [l for l in logs.split('\n') if 'ERROR' in l or 'FATAL' in l])
assert len(sc) == 15


In [ ]:
t0 = time.time()
r = subprocess.run([sys.executable, '/kaggle/working/code/exp_stats.py', '--eval', '/kaggle/working/eval', '--out', '/kaggle/working/stats', '--device', 'cuda:0'])
print('exit', r.returncode, 'min', round((time.time() - t0) / 60, 1))


In [ ]:
import json
for p in sorted(glob.glob('/kaggle/working/eval/set_0*/id_permutation_test.json')): print(p, open(p).read())
print(open('/kaggle/working/stats/summary.tsv').read())
R = json.load(open('/kaggle/working/stats/results.json'))
for k, c in R['per_cell'].items():
    print(k, c['role'], 'prev', round(c['prevalence_chance_ap'], 4), 'hours', round(c['hours'], 3), c['files_per_token'])
    for m, v in c['models'].items():
        print('   ', m, 'AP', round(v['ap_mean'], 4), 'MF1', round(v['macro_f1_mean'], 4), 'R@1', round(v['recall_at_1FAh_mean'], 4), 'FAh@1', round(v['achieved_FAh_at_1FAh_mean'], 2), 'R@5', round(v['recall_at_5FAh_mean'], 4), 'FAh@5', round(v['achieved_FAh_at_5FAh_mean'], 2))
print(json.dumps(R['holm_family'], indent=1))
